# Lake coverage - 2026-09-30

Plan 11 phase 9: one table of every dataset in the lake (source, known-on rule, first/last date, rows, keys),
the checks that found problems during plan 11 (start dates, per-call caps), and the cross-source quality views.
Reads the lake through the package; no network calls.

In [1]:
import pandas as pd

from quant_cn.core.config import Config
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.lake.release_lag_checker import ReleaseLagChecker

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
config = Config.load()
catalog = LakeCatalog(config.lake.root, config.datasets, read_only=True)
query = LakeQuery(catalog)

## 1. Every dataset
`first`/`last` are the known-on column's range (dates, months or report periods).

In [2]:
rows = []
for spec in config.datasets.values():
    view = f"curated.{spec.name}"
    if not query.has_view(view):
        rows.append({"dataset": spec.name, "source": spec.source, "status": "not curated"})
        continue
    col = spec.known_on or (spec.curated.partition_by or "")
    rng = f", min({col}) AS first, max({col}) AS last" if col else ""
    stats = query.sql(f"SELECT count(*) AS n_rows{rng} FROM {view}").to_dict("records")[0]
    rule = spec.known_on_rule + (f" +{spec.release_lag_days}d" if spec.release_lag_days else "")
    rule += f" ({spec.lag_sessions} sessions)" if spec.lag_sessions != 1 else ""
    rows.append({"dataset": spec.name, "source": spec.source, "sweep": spec.sweep.kind,
                 "known_on": spec.known_on or "-", "rule": rule, "start": config.get_start(spec.name),
                 **stats, "status": "ok"})
coverage = pd.DataFrame(rows)
coverage

,dataset,source,sweep,known_on,rule,start,n_rows,status,first,last
0,trade_cal,tushare,none,-,as_is,19901219,13467.0,ok,NaN,NaN
1,stock_basic,tushare,list_status,list_date,as_is,20190101,5908.0,ok,19901201,20260924
2,namechange,tushare,none,ann_date,as_is,20190101,14226.0,ok,19901201,20260929
3,index_daily,tushare,ts_code,trade_date,as_is,20100101,28357.0,ok,20100104,20260928
4,daily,tushare,trade_date,trade_date,as_is,20100101,14422449.0,ok,20100104,20260924
5,adj_factor,tushare,trade_date,trade_date,as_is,20100101,15079975.0,ok,20100104,20260924
6,daily_basic,tushare,trade_date,trade_date,as_is,20190101,8911483.0,ok,20190102,20260924
7,moneyflow,tushare,trade_date,trade_date,as_is,20190101,8663855.0,ok,20190102,20260924
8,stk_limit,tushare,trade_date,trade_date,as_is,20190101,9032953.0,ok,20190102,20260924
9,fina_indicator_vip,tushare,period,ann_date,as_is,20190101,284202.0,ok,20190408,20260928


## 2. Did every dataset reach its configured start?
A first date well after the configured start means the vendor has no earlier data, or a call was truncated
(D-070: `shibor` had silently started in 2018). Reference data without a date column is skipped.

In [3]:
dated = coverage.dropna(subset=["first"]).copy()
dated = dated[dated["first"].astype(str).str.len() == 8]
dated["gap_days"] = (pd.to_datetime(dated["first"]) - pd.to_datetime(dated["start"])).dt.days
dated.loc[dated["gap_days"] > 31, ["dataset", "source", "start", "first", "gap_days"]].sort_values("gap_days")

,dataset,source,start,first,gap_days
47,disclosure_date,tushare,20190101,20190329,87
9,fina_indicator_vip,tushare,20190101,20190408,97
10,income_vip,tushare,20190101,20190408,97
11,balancesheet_vip,tushare,20190101,20190408,97
12,cashflow_vip,tushare,20190101,20190408,97
44,express_vip,tushare,20190101,20190410,99
40,stk_surv,tushare,20190101,20210805,947


Expected late starts (vendor coverage, not truncation): `stk_surv` 2021-08, `broker_recommend` 2020-03,
`opt_daily` / `fut_*` / `margin` by design (explicit starts), `fx_daily` CFDs 2022-07 within a dataset that starts 2015.

## 3. Per-call caps (D-070, D-076, D-082)
A cap C means no single call returns more than C rows. Datasets whose largest key is a round hundred reached
on several keys are suspects; `index_weight` 12,000 is natural (1,000 constituents x 12 months).

In [4]:
query.sql('''
    WITH m AS (SELECT dataset, max(n_rows) AS max_rows FROM meta.fetch_log GROUP BY 1)
    SELECT f.dataset, m.max_rows, count(*) AS keys_at_max
    FROM meta.fetch_log f JOIN m USING (dataset)
    WHERE f.n_rows = m.max_rows AND m.max_rows % 100 = 0
    GROUP BY 1, 2 HAVING count(*) > 1 ORDER BY 3 DESC''')

,dataset,max_rows,keys_at_max
0,index_weight,12000,11


## 4. Release-lag rules vs observed releases (D-086)

In [5]:
ReleaseLagChecker(query, config.datasets).compute()

,dataset,rule_days,releases,observed_max,later_than_rule,latest
0,cn_cpi,20,179,20,0,20250910
1,cn_ppi,20,179,20,0,20250910
2,cn_pmi,7,100,7,0,20250831
3,cn_m,20,242,20,0,20250912


## 5. Cross-source quality (D-084, D-085)

In [6]:
for view in ["quality.margin_targets_agreement", "quality.price_crosscheck", "quality.adj_crosscheck"]:
    if not query.has_view(view):
        print(view, "- not built")
        continue
    if view.endswith("price_crosscheck"):
        out = query.sql(f'''SELECT substr(trade_date, 1, 4) AS yr, sum(compared) AS compared,
                                   sum(mismatches) AS mismatches, sum(mismatches) / sum(compared) AS rate
                            FROM {view} GROUP BY 1 ORDER BY 1''')
    elif view.endswith("adj_crosscheck"):
        out = query.sql(f'''SELECT count(*) AS events, count(*) FILTER (WHERE rel_diff > 1e-4) AS beyond_1e4,
                                   median(rel_diff) AS median_rel_diff, max(rel_diff) AS max_rel_diff FROM {view}''')
    else:
        out = query.sql(f"SELECT min(agreement) AS min_agreement, avg(agreement) AS mean_agreement, count(*) AS months FROM {view}")
    print(view); display(out)

quality.margin_targets_agreement


,min_agreement,mean_agreement,months
0,0.999486,0.999983,30


quality.price_crosscheck


,yr,compared,mismatches,rate
0,2019,884867.0,0.0,0.0
1,2020,945962.0,0.0,0.0
2,2021,1058863.0,0.0,0.0
3,2022,1146263.0,0.0,0.0
4,2023,1209466.0,0.0,0.0
5,2024,1233198.0,0.0,0.0
6,2025,1248108.0,0.0,0.0
7,2026,923638.0,0.0,0.0


quality.adj_crosscheck


,events,beyond_1e4,median_rel_diff,max_rel_diff
0,26565,6630,0.000023,4.975678


In [7]:
mism = "quality.price_mismatches"
if query.has_view(mism):
    display(query.sql(f'''SELECT ts_code, trade_date, tushare_close, baostock_close, rel_diff FROM {mism}
                          WHERE mismatch ORDER BY rel_diff DESC LIMIT 15'''))

,ts_code,trade_date,tushare_close,baostock_close,rel_diff


## 6. Derived view digests (byte-identical rebuild check)

In [8]:
import json
digests = json.loads((config.lake.root / "meta" / "manifest" / "derived_digests.json").read_text())
pd.Series(digests, name="md5").to_frame()

,md5
derived.fundamentals_long,6576db3fa27e8d2ccd025535ec06a25f
derived.fundamentals_states,c7b6697abd5f83c3fa3a5a17ce7fc954
derived.fut_daily_one_sided,4bbf2d24a8b51020e507e52b019f52ed
derived.fut_holding_brokers,5f8f0c800891b04a854637d91dd5cec3
derived.industry_l1,2b2e555c000bcd599731df19e905c21a
derived.margin_targets,3df6f4e8567069e6298551a898959aed
derived.prices_adj,af36a922d0735f432c153404539e9571
quality.adj_crosscheck,1cd71250757342fd0e673534ee6292cb
quality.margin_targets_agreement,42f5a331762397b51e813664227b5c92
quality.price_crosscheck,93a88e0b9833a52b0a3cead52ce3eb72


In [9]:
catalog.close()